In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

def transform_sales(df):
    product_schema = StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", DoubleType()),
        StructField("qty", IntegerType()),
        StructField("unit", StringType()),
    ])
    sales_df=df.withColumn('product',F.from_json('product',product_schema)).select(
        "customer_id",
        "customer_name",
        "product_name", 
        "order_date",
        "product_category",
        "product.curr",
        "product.id",
        "product.name",
        "product.price",
        "product.qty",
        "product.unit",
        "total_price",
        "last_updated_ts"
    )
    return sales_df
df=spark.read.table('retaildataplatform.bronze.s3sales_customers')
flatten_df=transform_sales(df)

# Deduplicate: keep latest record per customer_id for MERGE
_dedup_w = Window.partitionBy("customer_id").orderBy(F.col("last_updated_ts").desc())
flatten_df = flatten_df.withColumn("_rn", F.row_number().over(_dedup_w)).filter(F.col("_rn") == 1).drop("_rn") 

target_table="retaildataplatform.silver.sales"

if spark.catalog.tableExists("retaildataplatform.silver.sales"):
    print('Table Exists proceed with SCD-1 process.')
    target=DeltaTable.forName(spark,target_table)
    # SCD Type-1
    # Match in source and target --> Key exist in target
    (
    target.alias("target").merge(flatten_df.alias('source'),"target.customer_id=source.customer_id")
    # Update-> When records are Matching
    .whenMatchedUpdateAll()
    # Insert-> When records are not Matching
    .whenNotMatchedInsertAll()

    .execute()
        
    )
    

else:
    print('Table does not exist- First create Schema(Table)')
    spark.sql("create schema if not exists retaildataplatform.silver")
    flatten_df.write.format('delta').saveAsTable('retaildataplatform.silver.sales')